# MLP Cliff Research

This notebook is the working research log for the MLP cliff experiments. It is where we record the research questions, goals, assumptions, and findings. The operator-facing guidance for how experiment outputs are stored and maintained stays in `experiments/RESULTS.md`.

## Scope

- In scope: MLP-only experiments.
- Out of scope for now: GNN comparisons.
- Immediate goal: find whether MLP performance hits a clear cliff as evaluation level rises.
- Extension rule: if the best current MLP is still improving at Level 10, extend the evaluation ladder upward until the falloff becomes visible.

## Research Questions

1. For the current MLP family, where does win rate begin to collapse as level increases?
2. Does increasing `hidden_dim` or `num_layers` delay that cliff?
3. Are `batch_norm` or `residual` helpful after the main capacity sweep, or do they add complexity without moving the cliff?
4. How many evaluation games are required at each level so the 95% confidence interval is narrow enough to support decisions?
5. If Level 10 is still too easy for the best MLP, how far upward do we need to extend the ladder before the cliff appears?

## Working Hypotheses

- A larger MLP may push the cliff to a higher level, but only up to a point.
- The falloff will be easier to see in a level ladder than in one-off spot checks.
- Small evaluations such as 20 games are enough for smoke tests, but not enough for close architectural comparisons around the cliff.
- The most important region is the transition band where win rate is no longer near 0% or 100%. That is where the confidence interval is widest and where more games matter most.

## Current Experimental Design

- Train one MLP configuration at a time on a fixed archived dataset.
- Evaluate each trained checkpoint across a fixed level ladder. Start with Levels 2 through 10.
- If the strongest checkpoint is still comfortably winning at Level 10, extend the ladder upward instead of assuming the cliff is below 10.
- Keep the search budget fixed while sweeping architecture knobs.
- First-pass MLP sweep: `(128,1)`, `(256,2)`, `(512,3)`, `(1024,3)`.
- Second-pass feature check: test `batch_norm` and `residual` only on the best plain MLP.

## Evaluation Sizing Guidance

The experiment driver already reports a 95% Wilson confidence interval. For planning, the worst case is near a 50% win rate. Approximate worst-case half-widths are:

| Games | Approx. worst-case 95% half-width | Use |
| --- | --- | --- |
| 20 | +/- 20.1 points | Smoke test only |
| 40 | +/- 14.8 points | Very rough scouting |
| 60 | +/- 12.3 points | Still rough |
| 100 | +/- 9.6 points | Development comparisons |
| 150 | +/- 7.9 points | Better for cliff scouting |
| 200 | +/- 6.9 points | Strong development run |
| 300 | +/- 5.6 points | Confirmation-quality |
| 400 | +/- 4.9 points | Strong confirmation run |

Practical rule:

- Use 100 to 200 games for development sweeps.
- Use 300 to 400 games for confirmation runs near the suspected cliff.
- Spend the largest game budgets on the transition region, not on levels that are already obviously easy or impossible.

In [ ]:
import math

def wilson_interval_half_width(games: int, win_rate: float = 0.5, z: float = 1.96) -> float:
    """Return the Wilson 95% CI half-width for a given game count and win rate."""
    denominator = 1.0 + (z * z) / games
    center = (win_rate + (z * z) / (2.0 * games)) / denominator
    margin = (z * math.sqrt((win_rate * (1.0 - win_rate) / games) + (z * z) / (4.0 * games * games))) / denominator
    return margin

for games in [20, 40, 60, 100, 150, 200, 300, 400]:
    margin = wilson_interval_half_width(games)
    print(f'{games:>3} games -> +/- {margin * 100:.2f} points at worst case')

## Run Log Template

| Date | Config | Training Data | Eval Levels | Games/Level | Search Budget | Key Result | Notes |
| --- | --- | --- | --- | --- | --- | --- | --- |
| TBD | hd256_nl2 | archived ??? | 2-10 | 100 | sims=?, depth=?, cpuct=? | TBD | Baseline |

In [ ]:
from pathlib import Path
import json

REPO_ROOT = Path.cwd()
ARTIFACTS_ROOT = REPO_ROOT / 'experiments' / 'runtime' / 'artifacts'

def load_evaluation_shards():
    records = []
    for path in ARTIFACTS_ROOT.glob('*/**/evaluation_shards.jsonl'):
        with path.open('r', encoding='utf-8') as handle:
            for line in handle:
                records.append(json.loads(line))
    return sorted(
        records,
        key=lambda row: (
            str(row.get('experiment_id', '')),
            str(row.get('run_id', '')),
            int(row.get('level', 0)),
            int(row.get('shard_index', 0)),
        ),
    )

evaluation_records = load_evaluation_shards()
print(f'Loaded {len(evaluation_records)} shard rows from {ARTIFACTS_ROOT}')
evaluation_records[:2]

## Plot Completed Runs

Use the cells below to aggregate completed experiment-driver runs and visualize win rate across levels with Wilson confidence intervals. Set `FOCUS_PREFIX` to the experiment family you want to inspect, such as `mlp-cliff-`.

In [ ]:
from collections import defaultdict
import importlib.util
import math

def wilson_interval_percent(wins, games, z=1.96):
    if games <= 0:
        return 0.0, 0.0
    win_ratio = wins / games
    denominator = 1.0 + (z * z) / games
    center = (win_ratio + (z * z) / (2.0 * games)) / denominator
    margin = (
        z
        * math.sqrt((win_ratio * (1.0 - win_ratio) / games) + (z * z) / (4.0 * games * games))
        / denominator
    )
    return 100.0 * (center - margin), 100.0 * (center + margin)

def aggregate_ladder_rows(records, experiment_prefix=None):
    grouped = defaultdict(
        lambda: {
            'games': 0,
            'wins': 0,
            'weighted_moves': 0.0,
            'weighted_time': 0.0,
        }
    )
    metadata = {}

    for row in records:
        experiment_id = str(row.get('experiment_id', ''))
        if experiment_prefix and not experiment_id.startswith(experiment_prefix):
            continue

        key = (
            str(row.get('run_id', '')),
            experiment_id,
            int(row.get('level', 0)),
        )
        games = int(row.get('games_tested', 0))
        wins = int(row.get('games_won', 0))
        grouped[key]['games'] += games
        grouped[key]['wins'] += wins
        grouped[key]['weighted_moves'] += float(row.get('avg_moves', 0.0)) * games
        grouped[key]['weighted_time'] += float(row.get('avg_time_seconds', 0.0)) * games
        metadata[key] = {
            'checkpoint_path': row.get('checkpoint_path'),
            'architecture_family': row.get('architecture_family'),
        }

    rows = []
    for (run_id, experiment_id, level), aggregate in grouped.items():
        games = aggregate['games']
        wins = aggregate['wins']
        ci_low, ci_high = wilson_interval_percent(wins, games)
        win_percent = 100.0 * wins / games if games else 0.0
        rows.append(
            {
                'run_id': run_id,
                'experiment_id': experiment_id,
                'level': level,
                'games': games,
                'wins': wins,
                'win_percent': win_percent,
                'ci_low': ci_low,
                'ci_high': ci_high,
                'avg_moves': aggregate['weighted_moves'] / games if games else 0.0,
                'avg_time_seconds': aggregate['weighted_time'] / games if games else 0.0,
                'checkpoint_path': metadata[(run_id, experiment_id, level)]['checkpoint_path'],
                'architecture_family': metadata[(run_id, experiment_id, level)]['architecture_family'],
            }
        )

    return sorted(rows, key=lambda row: (row['experiment_id'], row['run_id'], row['level']))

FOCUS_PREFIX = 'mlp-cliff-'
ladder_rows = aggregate_ladder_rows(evaluation_records, experiment_prefix=FOCUS_PREFIX)
print(f'Aggregated {len(ladder_rows)} run-level rows for prefix {FOCUS_PREFIX!r}')
ladder_rows[:5]

if not ladder_rows:
    print('No matching completed runs yet. Run the mlp-cliff specs first or change FOCUS_PREFIX.')
elif importlib.util.find_spec('matplotlib') is None:
    print('matplotlib is not installed in this environment. Install it to render the plot, or inspect ladder_rows directly.')
else:
    import matplotlib.pyplot as plt

    series_by_run = defaultdict(list)
    for row in ladder_rows:
        series_by_run[row['run_id']].append(row)

    fig, ax = plt.subplots(figsize=(10, 6))
    for run_id, rows in sorted(series_by_run.items()):
        ordered_rows = sorted(rows, key=lambda row: row['level'])
        levels = [row['level'] for row in ordered_rows]
        win_percents = [row['win_percent'] for row in ordered_rows]
        lower_errors = [row['win_percent'] - row['ci_low'] for row in ordered_rows]
        upper_errors = [row['ci_high'] - row['win_percent'] for row in ordered_rows]
        label = ordered_rows[0]['experiment_id'] or run_id
        ax.errorbar(
            levels,
            win_percents,
            yerr=[lower_errors, upper_errors],
            marker='o',
            capsize=3,
            linewidth=1.5,
            label=label,
        )

    ax.set_title('MLP cliff sweep: win rate by level with 95% Wilson confidence intervals')
    ax.set_xlabel('Evaluation level')
    ax.set_ylabel('Win rate (%)')
    ax.set_ylim(-5, 105)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(title='Experiment', bbox_to_anchor=(1.02, 1.0), loc='upper left')
    plt.tight_layout()
    plt.show()